# Preprocessing

Nettoyage, split, features, anti-leakage.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[0]
sys.path.append(str(ROOT / 'src'))

import pandas as pd

from data.download import download_raw_dataset
from data.preprocess import preprocess_and_split


In [ ]:
csv_path = download_raw_dataset()
splits, preprocess_path = preprocess_and_split(Path(csv_path))
splits, preprocess_path


(SplitPaths(train=PosixPath('/home/narcisse/Projets/detecter-fraude-bancaire/data/processed/train.parquet'), val=PosixPath('/home/narcisse/Projets/detecter-fraude-bancaire/data/processed/val.parquet'), test=PosixPath('/home/narcisse/Projets/detecter-fraude-bancaire/data/processed/test.parquet')),
 PosixPath('/home/narcisse/Projets/detecter-fraude-bancaire/data/processed/preprocess.joblib'))

In [ ]:
train_df = pd.read_parquet(splits.train)
val_df = pd.read_parquet(splits.val)
test_df = pd.read_parquet(splits.test)
train_df.head()


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V22,V23,V24,V25,V26,V27,V28,Amount,Amount_log1p,Class
0,-0.206877,-0.185825,0.287991,-0.944344,-1.263563,0.028440,-1.779481,0.991521,0.696680,-1.272164,...,-1.159025,-1.929750,-1.316727,0.371068,-0.599900,0.721683,-0.180988,-0.295427,0.073301,0
1,1.453255,0.112436,0.741913,0.974605,2.898725,-0.170958,0.450444,0.549271,-2.460040,-0.712111,...,-0.287039,0.459242,-0.935088,0.244134,-1.112502,-0.561676,-0.107899,1.207768,0.921746,0
2,-0.582168,-0.151862,-0.182083,0.397831,0.086948,-1.094136,0.760249,1.848832,-0.722309,0.250628,...,-0.841520,0.510289,0.893086,-1.346545,1.619791,-0.580995,1.566062,-0.716439,-0.330296,0
3,1.615440,-1.072460,-1.858867,-0.067159,-0.776549,0.469348,0.697005,-0.231688,-1.130973,0.511967,...,1.743851,-0.250404,-0.218495,-1.070282,0.915852,2.541979,-1.307045,-1.149562,-0.961061,0
4,0.610780,-0.236916,-2.059129,1.226311,0.580972,-0.465109,-0.836525,-1.667224,1.395260,1.613063,...,-0.606664,-2.236952,0.056684,0.175228,0.046149,-0.427666,1.283930,1.629519,1.082141,0


In [ ]:
train_df['Class'].value_counts(normalize=True), val_df['Class'].value_counts(normalize=True), test_df['Class'].value_counts(normalize=True)


(Class
 0    0.979971
 1    0.020029
 Name: proportion, dtype: float64,
 Class
 0    0.98
 1    0.02
 Name: proportion, dtype: float64,
 Class
 0    0.9801
 1    0.0199
 Name: proportion, dtype: float64)

In [ ]:
# Anti-leakage: ensure no duplicate rows across splits
train_hash = pd.util.hash_pandas_object(train_df, index=False)
val_hash = pd.util.hash_pandas_object(val_df, index=False)
test_hash = pd.util.hash_pandas_object(test_df, index=False)
len(set(train_hash) & set(val_hash)), len(set(train_hash) & set(test_hash)), len(set(val_hash) & set(test_hash))


(0, 0, 0)

## Checks anti data leakage

- Split stratifié avant fit du preprocess.
- Pipeline fit uniquement sur train.
- Aucune cible utilisée dans les features.
- Vérification de doublons entre splits.